<a href="https://colab.research.google.com/github/adanielce/IA_S8/blob/main/PROYECTO_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install roboflow ultralytics

Esta celda instala las librerías necesarias `roboflow` y `ultralytics`, esenciales para descargar conjuntos de datos y trabajar con modelos YOLO para la detección de objetos.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="kX5HXsbw0XDLFWwSiKiJ")
project = rf.workspace("yogitas-workspace-p3oyi").project("pothole-detection-ehtnh")
version = project.version(1)
dataset = version.download("yolov8")

print(f"El dataset se guardó en: {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pothole-Detection-1 in yolov8:: 100%|██████████| 12429/12429 [00:01<00:00, 6430.10it/s]


El dataset se guardó en: /content/Pothole-Detection-1


Esta celda inicializa Roboflow con tu clave API, descarga una versión específica del conjunto de datos 'Pothole-Detection' e imprime su ubicación de almacenamiento local. Este conjunto de datos se utilizará para entrenar el modelo YOLO.

In [ ]:
from ultralytics import YOLO

# 1. Cargar el modelo base (YOLOv8 'nano' es el más rápido y ligero para empezar)
model = YOLO("yolov8n.pt")

# 2. Iniciar el entrenamiento
# Le pasamos la ubicación exacta del archivo data.yaml que descargó Roboflow
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=25,           # Número de vueltas de entrenamiento (25 es un buen inicio)
    imgsz=640,           # Resolución a la que redimensionará las imágenes
    batch=16,            # Cuántas imágenes procesa al mismo tiempo la GPU
    project="Baches_Chilpancingo", # Nombre de la carpeta principal de resultados
    name="prueba_inicial"          # Nombre de esta ejecución
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Pothole-Detection-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=prueba_inicial, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

Esta celda carga un modelo YOLOv8 'nano' pre-entrenado y luego inicia el proceso de entrenamiento. Especifica la ubicación del conjunto de datos, el número de épocas de entrenamiento, el tamaño de la imagen, el tamaño del lote y los nombres de proyecto/ejecución para organizar los resultados.

In [ ]:
# Instalación de librerías esenciales para visión artificial y análisis de datos
!pip install ultralytics pandas matplotlib seaborn roboflow

import os
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from ultralytics import YOLO

# Creación de la estructura de directorios para el proyecto
directorios = ['datos_entrada', 'evidencias', 'modelos']
for dir_name in directorios:
    os.makedirs(dir_name, exist_ok=True)

print("Entorno configurado correctamente. Directorios creados.")

Entorno configurado correctamente. Directorios creados.


Esta celda asegura que todas las librerías requeridas para visión por computadora y análisis de datos estén instaladas e importadas. También establece una estructura de directorios (`datos_entrada`, `evidencias`, `modelos`) para organizar los datos de entrada, los resultados de detección y los modelos entrenados.

In [ ]:
# Inicialización del modelo con los pesos sinápticos obtenidos en la fase de entrenamiento
ruta_modelo = '/content/runs/detect/Baches_Chilpancingo/prueba_inicial/weights/best.pt'

try:
    modelo_baches = YOLO(ruta_modelo)
    print("Modelo YOLOv8 cargado exitosamente en memoria.")
except Exception as e:
    print(f"Error al cargar el modelo. Verifica que best.pt esté en la ruta correcta. Detalles: {e}")

Modelo YOLOv8 cargado exitosamente en memoria.


Esta celda carga en memoria el modelo YOLOv8 entrenado con mejor rendimiento desde la ruta especificada. Este modelo se utilizará para detectar baches en nuevas imágenes y videos.

In [ ]:
# Coordenadas base simuladas (Ej. Centro de Chilpancingo)
LAT_BASE = 17.5512
LNG_BASE = -99.5014

def procesar_y_registrar(ruta_archivo, archivo_csv='/content/evidencias/registro_baches.csv'):
    nombre_archivo = os.path.basename(ruta_archivo)

    # Inferencia con YOLOv8 (Umbral de confianza del 50%)
    resultados = modelo_baches.predict(source=ruta_archivo, conf=0.50, save=False)

    registros_actuales = []

    for r in resultados:
        # Generar evidencia visual
        im_array = r.plot()  # Dibuja los bounding boxes y confianzas
        ruta_salida = f"/content/evidencias/anotado_{nombre_archivo}"
        cv2.imwrite(ruta_salida, im_array)

        # Extraer datos de la matriz de resultados
        cajas = r.boxes
        for caja in cajas:
            confianza = float(caja.conf[0])
            coordenadas = caja.xyxy[0].tolist() # [x1, y1, x2, y2]

            # Simulación de desplazamiento GPS para cada detección
            lat_simulada = LAT_BASE + np.random.uniform(-0.005, 0.005)
            lng_simulada = LNG_BASE + np.random.uniform(-0.005, 0.005)

            registro = {
                'Fecha': datetime.now().strftime('%Y-%m-%d'),
                'Hora': datetime.now().strftime('%H:%M:%S'),
                'Archivo': nombre_archivo,
                'Confianza (%)': round(confianza * 100, 2),
                'Coordenadas_BBox': f"[{round(coordenadas[0])}, {round(coordenadas[1])}, {round(coordenadas[2])}, {round(coordenadas[3])}]",
                'Latitud': lat_simulada,
                'Longitud': lng_simulada
            }
            registros_actuales.append(registro)

    # Persistencia en archivo CSV
    df_nuevo = pd.DataFrame(registros_actuales)
    if os.path.exists(archivo_csv):
        df_existente = pd.read_csv(archivo_csv)
        df_final = pd.concat([df_existente, df_nuevo], ignore_index=True)
    else:
        df_final = df_nuevo

    df_final.to_csv(archivo_csv, index=False)
    print(f"Procesamiento completo: {nombre_archivo}. {len(registros_actuales)} baches registrados.")

Esta celda define la función `procesar_y_registrar`, que toma la ruta de una imagen, realiza la detección de baches utilizando el modelo YOLO cargado, genera evidencia visual con cuadros delimitadores, simula coordenadas GPS para cada detección y registra los resultados (fecha, hora, confianza, coordenadas) en un archivo CSV.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def generar_dashboard():
    ruta_csv = '/content/evidencias/registro_baches.csv'
    if not os.path.exists(ruta_csv):
        print("No hay datos registrados para generar el dashboard.")
        return

    df = pd.read_csv(ruta_csv)

    # Configuración de la figura principal
    plt.figure(figsize=(15, 5))
    sns.set_theme(style="whitegrid")

    # Gráfica 1: Distribución de Niveles de Confianza
    plt.subplot(1, 2, 1)
    sns.histplot(df['Confianza (%)'], bins=10, kde=True, color='darkblue')
    plt.title('Distribución de la Confianza de Detección')
    plt.xlabel('Confianza (%)')
    plt.ylabel('Frecuencia')
    # Línea roja punteada para marcar el promedio
    plt.axvline(df['Confianza (%)'].mean(), color='red', linestyle='--', label=f'Media: {df["Confianza (%)"].mean():.1f}%')
    plt.legend()

    # Gráfica 2: Detecciones por Archivo
    plt.subplot(1, 2, 2)
    conteo_archivos = df['Archivo'].value_counts()
    sns.barplot(x=conteo_archivos.index, y=conteo_archivos.values, palette='viridis')
    plt.title('Total de Baches Detectados por Archivo')
    plt.xlabel('Nombre de Archivo')
    plt.ylabel('Cantidad de Detecciones')
    plt.xticks(rotation=45)

    # Ajustar espacios para que no se empalmen los textos
    plt.tight_layout()
    plt.show()

    # Imprimir métricas resumidas en texto debajo de las gráficas
    print("--- RESUMEN EJECUTIVO ---")
    print(f"Total de baches reportados: {len(df)}")
    print(f"Confianza promedio del sistema: {df['Confianza (%)'].mean():.2f}%")

# Llamada a la función para que se ejecute al correr la celda
generar_dashboard()

No hay datos registrados para generar el dashboard.


Esta celda define y llama inmediatamente a la función `generar_dashboard`. Esta función lee los datos de baches detectados del CSV, visualiza la distribución de la confianza de detección y muestra el número total de detecciones por archivo procesado utilizando `matplotlib` y `seaborn`.

In [ ]:
# Celda 5: Inferencia y Registro con fotos reales de tu calle
import os

carpeta_pruebas = '/content/datos_entrada'

try:
    # Filtramos para asegurarnos de leer solo archivos de imagen o video
    archivos_prueba = [f for f in os.listdir(carpeta_pruebas) if f.endswith(('.jpg', '.jpeg', '.png', '.mp4'))]

    if len(archivos_prueba) > 0:
        print(f"Iniciando inferencia en el mundo real procesando {len(archivos_prueba)} archivos...")
        for archivo in archivos_prueba:
            ruta_completa = os.path.join(carpeta_pruebas, archivo)
            procesar_y_registrar(ruta_completa)

        print("\n¡Análisis completado! Actualizando dashboard con datos de tu calle...")
        generar_dashboard()
    else:
        print("La carpeta 'datos_entrada' está vacía. Sube tus fotos de las calles antes de ejecutar.")

except FileNotFoundError:
    print(f"Error: No se encontró la ruta {carpeta_pruebas}.")

La carpeta 'datos_entrada' está vacía. Sube tus fotos de las calles antes de ejecutar.


Esta celda procesa archivos de imagen o video ubicados en la carpeta `/content/datos_entrada`. Itera sobre cada archivo, aplica la función `procesar_y_registrar` para la detección y luego actualiza el panel de control con los nuevos datos del mundo real. También proporciona un mensaje si la carpeta de entrada está vacía.

In [ ]:
# Celda 6: Visualizador de Evidencias
import cv2
import matplotlib.pyplot as plt
import os

carpeta_evidencias = '/content/evidencias'

# Buscamos las imágenes que el sistema guardó con el prefijo 'anotado_'
imagenes_anotadas = [f for f in os.listdir(carpeta_evidencias) if f.startswith('anotado_') and f.endswith(('.jpg', '.jpeg', '.png'))]

# Mostramos solo 3 para no saturar la pantalla (puedes cambiar este número a 5 o 10 si quieres ver más)
cantidad_a_mostrar = min(3, len(imagenes_anotadas))

if cantidad_a_mostrar > 0:
    print(f"--- MOSTRANDO {cantidad_a_mostrar} EJEMPLOS DE DETECCIÓN VISUAL ---")

    # Configuramos el tamaño del lienzo
    plt.figure(figsize=(18, 6))

    for i in range(cantidad_a_mostrar):
        ruta_img = os.path.join(carpeta_evidencias, imagenes_anotadas[i])

        # Leemos la imagen con OpenCV y ajustamos los colores para Matplotlib
        img = cv2.imread(ruta_img)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Dibujamos la imagen en su respectivo espacio
        plt.subplot(1, cantidad_a_mostrar, i+1)
        plt.imshow(img_rgb)
        plt.axis('off') # Ocultamos los ejes para que se vea más limpio
        plt.title(f"Evidencia {i+1}\n{imagenes_anotadas[i][:15]}...")

    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron imágenes procesadas en la carpeta de evidencias.")

No se encontraron imágenes procesadas en la carpeta de evidencias.


Esta celda sirve como un visualizador de evidencias. Busca imágenes anotadas (con el prefijo 'anotado_') en la carpeta `/content/evidencias` y muestra hasta tres ejemplos para exhibir las capacidades de detección del modelo.

///////////////////////////////////////////////////////////////////////////////////////////

Celda 1: Configuración Inicial del Entorno
Texto de contexto (Celda de texto en Colab):

1. Instalación de Dependencias y Estructura de Directorios
En esta sección se instalan las librerías necesarias para el análisis de datos (Pandas, Seaborn), visión por computadora (Ultralytics, OpenCV) y se crean las carpetas virtuales donde el sistema alojará las imágenes de prueba, las evidencias procesadas y los modelos generados.

Código (Celda de código):

In [ ]:
%%writefile app.py
import streamlit as st
from ultralytics import YOLO
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from datetime import datetime
import os
from streamlit_webrtc import webrtc_streamer, RTCConfiguration
import av

# --- RUTAS ABSOLUTAS DE COLAB ---
RUTA_MODELO = '/content/runs/detect/Baches_Chilpancingo/prueba_inicial/weights/best.pt'
RUTA_CSV = '/content/evidencias/registro_baches.csv'
RUTA_RESULTS = '/content/runs/detect/Baches_Chilpancingo/prueba_inicial/results.png'
RUTA_MATRIX = '/content/runs/detect/Baches_Chilpancingo/prueba_inicial/confusion_matrix.png'

LAT_BASE = 17.5512
LNG_BASE = -99.5014

st.set_page_config(page_title="Monitor Vial IA", page_icon="🚧", layout="wide")

@st.cache_resource
def cargar_modelo():
    try:
        return YOLO(RUTA_MODELO)
    except Exception as e:
        return None

modelo_baches = cargar_modelo()

st.title("🚧 Sistema Inteligente de Inspección de Pavimento")
st.markdown("Plataforma web para detección de anomalías viales mediante Visión Artificial.")

if modelo_baches is None:
    st.error(f"Error: No se encontró el modelo en {RUTA_MODELO}.")
    st.stop()

# --- NUEVA ESTRUCTURA DE 3 PESTAÑAS ---
tab1, tab2, tab3 = st.tabs(["🔍 Inspección por Imagen", "🎥 Cámara en Tiempo Real", "📊 Dashboard Analítico"])

# ==========================================
# PESTAÑA 1: IMÁGENES ESTÁTICAS
# ==========================================
with tab1:
    st.header("Analizador Estático de Daños Viales")
    archivo_subido = st.file_uploader("Carga una fotografía del asfalto...", type=["jpg", "jpeg", "png"])

    if archivo_subido is not None:
        imagen_pil = Image.open(archivo_subido)
        imagen_cv = cv2.cvtColor(np.array(imagen_pil), cv2.COLOR_RGB2BGR)
        alto_img, ancho_img, _ = imagen_cv.shape
        area_total = alto_img * ancho_img

        resultados = modelo_baches.predict(source=imagen_cv, conf=0.50, save=False)
        r = resultados[0]

        imagen_anotada_rgb = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
        st.image(imagen_anotada_rgb, caption='Detección de YOLOv8', use_container_width=True)

        cajas = r.boxes
        if len(cajas) > 0:
            st.success(f"⚠️ Se detectaron {len(cajas)} bache(s).")
            registros_actuales = []
            cols = st.columns(len(cajas))

            for i, caja in enumerate(cajas):
                confianza = float(caja.conf[0])
                coords = caja.xyxy[0].tolist()
                area_bache = (coords[2] - coords[0]) * (coords[3] - coords[1])
                pct_area = (area_bache / area_total) * 100

                if pct_area > 15.0: riesgo = "🔴 CRÍTICO"
                elif pct_area > 5.0: riesgo = "🟠 MEDIO"
                else: riesgo = "🟡 BAJO"

                with cols[i]:
                    st.markdown(f"**Bache #{i+1}**")
                    st.write(f"Certeza: {confianza * 100:.2f}% | Área: {pct_area:.2f}%")
                    st.write(riesgo)

                registros_actuales.append({
                    'Fecha': datetime.now().strftime('%Y-%m-%d'),
                    'Hora': datetime.now().strftime('%H:%M:%S'),
                    'Archivo': archivo_subido.name,
                    'Confianza (%)': round(confianza * 100, 2),
                    'Riesgo Estimado': riesgo.split(" ")[0],
                    'Coordenadas_BBox': f"[{round(coords[0])}, {round(coords[1])}, {round(coords[2])}, {round(coords[3])}]",
                    'Latitud': LAT_BASE + np.random.uniform(-0.005, 0.005),
                    'Longitud': LNG_BASE + np.random.uniform(-0.005, 0.005)
                })

            df_nuevo = pd.DataFrame(registros_actuales)
            df_final = pd.concat([pd.read_csv(RUTA_CSV), df_nuevo]) if os.path.exists(RUTA_CSV) else df_nuevo

            # --- CORRECCIÓN CLAVE: Crear la carpeta si no existe antes de guardar ---
            os.makedirs(os.path.dirname(RUTA_CSV), exist_ok=True)

            df_final.to_csv(RUTA_CSV, index=False)
            st.caption("Datos guardados en la base de datos central.")
        else:
            st.success("✅ Asfalto en óptimas condiciones.")

# ==========================================
# PESTAÑA 2: CÁMARA EN VIVO (WebRTC con TURN)
# ==========================================
with tab2:
    st.header("Detección de Baches en Tiempo Real (Video)")
    st.markdown("Permite el acceso a tu cámara web. El sistema dibujará los polígonos/cajas delimitadoras en vivo.")

    class ProcesadorVideo:
        def recv(self, frame):
            img = frame.to_ndarray(format="bgr24")
            resultados = modelo_baches.predict(source=img, conf=0.45, save=False, verbose=False)
            img_anotada = resultados[0].plot()
            return av.VideoFrame.from_ndarray(img_anotada, format="bgr24")

    configuracion_rtc = RTCConfiguration({
        "iceServers": [
            {"urls": ["stun:stun.l.google.com:19302"]},
            {
                "urls": ["turn:openrelay.metered.ca:80", "turn:openrelay.metered.ca:443?transport=tcp"],
                "username": "openrelayproject",
                "credential": "openrelayproject"
            }
        ]
    })

    webrtc_streamer(
        key="detector_baches",
        video_processor_factory=ProcesadorVideo,
        rtc_configuration=configuracion_rtc,
        media_stream_constraints={"video": True, "audio": False}
    )

    st.info("💡 Consejo: Apunta la cámara a una fotografía de un bache en tu celular para probar el funcionamiento en vivo.")

# ==========================================
# PESTAÑA 3: DASHBOARD
# ==========================================
with tab3:
    st.header("Métricas y Rendimiento")
    if os.path.exists(RUTA_CSV):
        df_dash = pd.read_csv(RUTA_CSV)
        fig = plt.figure(figsize=(12, 4))
        sns.set_theme(style="whitegrid")

        plt.subplot(1, 2, 1)
        sns.histplot(df_dash['Confianza (%)'], bins=10, kde=True, color='darkblue')
        plt.title('Distribución de Confianza')
        plt.axvline(df_dash['Confianza (%)'].mean(), color='red', linestyle='--')

        plt.subplot(1, 2, 2)
        conteo = df_dash['Archivo'].value_counts()
        sns.barplot(x=conteo.index, y=conteo.values, palette='viridis')
        plt.title('Total de Baches Detectados por Archivo')
        plt.xticks(rotation=45)

        plt.tight_layout()
        st.pyplot(fig)
    else:
        st.info("Sube una imagen estática primero para inicializar el Dashboard.")

    st.divider()
    col1, col2 = st.columns(2)
    with col1:
        st.write("**Curvas de Entrenamiento (Loss / mAP)**")
        if os.path.exists(RUTA_RESULTS):
            st.image(RUTA_RESULTS, caption="Gráficas detalladas del proceso de entrenamiento", use_container_width=True)
    with col2:
        st.write("**Matriz de Confusión**")
        if os.path.exists(RUTA_MATRIX):
            st.image(RUTA_MATRIX, caption="Matriz de Confusión: Evaluación de aciertos y errores", use_container_width=True)

Overwriting app.py


Esta celda crea una aplicación web Streamlit (`app.py`) que proporciona una interfaz interactiva para la detección de baches. Incluye funciones para el análisis de imágenes estáticas, inspección con cámara en tiempo real utilizando WebRTC y un panel analítico. Precarga el modelo YOLO e integra el registro de datos.

In [ ]:
import subprocess, time, os

# 1. Aniquilar cualquier proceso zombi rebelde a la fuerza
os.system("killall -9 cloudflared")
os.system("killall -9 streamlit")
os.system("fuser -k 8501/tcp")
time.sleep(3)

# 2. Levantar la página web
print("Iniciando el motor web...")
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.address', '0.0.0.0'])

# 3. Darle 10 segundos a Streamlit para que cargue todas las librerías de IA
print("Cargando Inteligencia Artificial... Espera 10 segundos...")
time.sleep(10)

# 4. Lanzar el túnel de Cloudflare
print("====================================================================")
print("⏳ Busca el enlace que termina en '.trycloudflare.com'")
print("⚠️ IMPORTANTE: Lee las instrucciones abajo antes de darle clic.")
print("====================================================================")
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8501

Iniciando el motor web...
Cargando Inteligencia Artificial... Espera 10 segundos...
⏳ Busca el enlace que termina en '.trycloudflare.com'
⚠️ IMPORTANTE: Lee las instrucciones abajo antes de darle clic.
2026-06-02T20:21:58Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-02T20:21:58Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-02T20:22:03Z INF +--------------------------------------------------------------------------------------------+

Esta celda inicia la aplicación Streamlit y crea un túnel de acceso público a la web utilizando Cloudflare. Primero se asegura de que cualquier proceso anterior sea terminado, luego lanza el servidor Streamlit, espera a que se inicialice y finalmente establece el túnel de Cloudflare, proporcionando una URL compartible para acceder a la aplicación web.